# 5 · MMLU regression check (LightEval)

> ⚠️ **UNDER CONSTRUCTION — not working yet.** Scaffold that still needs environment-specific tuning; don't rely on its output.

**Purpose: a regression check, not the tool-calling value metric.** MMLU is multiple-choice world knowledge. We run it on base / SFT / SFT+GRPO to confirm tool-calling fine-tuning **didn't degrade** general knowledge — you want the three roughly on par, not a big drop.

Runs as one GPU job (`eval_mmlu.sh` → LightEval on each checkpoint), uploads to `{s3_base}/mmlu`, then downloads + shows the summary.

> LightEval's task string / CLI flags vary by version. If the job errors on the task string, change `MMLU_TASKS` below (and see `src/eval_mmlu.py::run_lighteval`).

In [ ]:
import os
try:
    os.getcwd()
except (FileNotFoundError, OSError):
    os.chdir(os.path.expanduser('~')); print('cwd was invalid; switched to', os.getcwd())
import json
from sagemaker.core import image_uris
try:
    from sagemaker.core.helper.session_helper import Session, get_execution_role
except Exception:
    from sagemaker.session import Session
    from sagemaker import get_execution_role
from sagemaker.train.model_trainer import ModelTrainer
from sagemaker.train.configs import SourceCode, Compute, StoppingCondition

sess = Session()
cfg = json.load(open('toolcall_config.json'))
region = cfg['region']
INSTANCE_TYPE = cfg['gpu_instance']
image_uri = cfg['gpu_image_uri'] or image_uris.retrieve(
    framework='pytorch', region=region, version=cfg['pt_version'],
    instance_type=INSTANCE_TYPE, image_scope='training')
print('Region  :', region)
print('Instance:', INSTANCE_TYPE)
print('Image   :', image_uri)

In [ ]:
RESULTS_S3 = cfg['s3_base'] + '/mmlu'
environment = {
    'BASE_MODEL': cfg['base_model'],
    'SFT_MODEL_S3': cfg['sft_model_s3'],
    'GRPO_MODEL_S3': cfg['grpo_model_s3'],
    'OUTPUT_RESULTS_S3': RESULTS_S3,
    'EVAL_BASE': str(cfg.get('eval_base', 1)),
    'MMLU_TASKS': 'leaderboard|mmlu|5|0',   # 5-shot MMLU; edit per lighteval version
    'HF_HOME': '/opt/ml/hf_home', 'HF_HUB_ENABLE_HF_TRANSFER': '1', 'TOKENIZERS_PARALLELISM': 'false',
}
if cfg.get('hf_token'):
    environment['HF_TOKEN'] = cfg['hf_token']
    environment['HUGGING_FACE_HUB_TOKEN'] = cfg['hf_token']
source_code = SourceCode(source_dir='../src', requirements='requirements.txt', entry_script='eval_mmlu.sh')
trainer = ModelTrainer(sagemaker_session=sess, role=cfg['role'], training_image=image_uri,
    source_code=source_code, compute=Compute(instance_type=INSTANCE_TYPE, instance_count=1),
    environment=environment, base_job_name='toolcall-mmlu',
    stopping_condition=StoppingCondition(max_runtime_in_seconds=12*3600))
trainer.train(wait=True)
print('MMLU results ->', RESULTS_S3)

### Download + show

In [ ]:
import boto3, json, os
from urllib.parse import urlparse
u = urlparse(RESULTS_S3); bucket, prefix = u.netloc, u.path.strip('/')
s3 = boto3.client('s3', region_name=region)
os.makedirs('eval_results', exist_ok=True)
paginator = s3.get_paginator('list_objects_v2')
got = []
for page in paginator.paginate(Bucket=bucket, Prefix=prefix + '/'):
    for obj in page.get('Contents', []):
        key = obj['Key']; rel = key[len(prefix)+1:]
        if not rel or rel.endswith('/'):
            continue
        dest = os.path.join('eval_results', rel)
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        s3.download_file(bucket, key, dest); got.append(rel)
print(f'downloaded {len(got)} files to eval_results/')

In [ ]:
import json, pandas as pd
summ = json.load(open('eval_results/mmlu_summary.json'))
df = pd.DataFrame(summ).set_index('model')
display(df)
print('\nRegression check: SFT / SFT+GRPO mmlu_acc should be close to base.')